In [ ]:
# Cài đặt các thư viện cốt lõi và thư viện bổ sung phục vụ cấu trúc WSSS nâng cao
!pip install kagglehub opencv-python pydensecrf medpy torchmetrics

# Lưu ý: Nếu cài đặt pydensecrf trên Colab gặp lỗi biên dịch, hãy uncomment dòng dưới đây để cài từ source:
# !pip install git+https://github.com/lucasb-eyer/pydensecrf.git

In [ ]:
from google.colab import drive
import kagglehub

# 1. Mount Google Drive để lưu checkpoint và kết quả huấn luyện
drive.mount('/content/drive')

# 2. Tải dataset tự động từ Kaggle về thư mục cache hệ thống
data_path = kagglehub.dataset_download("mahmudulhasantasin/fracatlas-original-dataset")
print(f"Dataset đã được tải về cấu trúc thư mục: {data_path}")

In [ ]:
# Di chuyển vào thư mục chứa mã nguồn của bạn trên Drive
# (Hãy thay đổi đường dẫn dưới đây cho đúng với cấu trúc thư mục repo của bạn trên Drive)
%cd /content/drive/MyDrive/Colab Notebooks/Thesis/project

In [ ]:
# Di chuyển vào thư mục chứa mã nguồn của bạn trên Drive
# (Hãy thay đổi đường dẫn dưới đây cho đúng với cấu trúc thư mục repo của bạn trên Drive)
%cd /content/drive/MyDrive/Colab Notebooks/Thesis/project

In [ ]:
%%writefile generate_pseudo_masks.py
from __future__ import annotations

import pydensecrf.densecrf as dcrf
from pydensecrf.utils import create_pairwise_bilateral, create_pairwise_gaussian, unary_from_softmax

import argparse
import sys
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

ROOT = Path(__file__).resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from datasets.fracatlas import FracAtlasClassificationDataset
from models.classifier import DenseNet121AnatomyClassifier
from models.gradcam import GradCAM
from pseudo.cam_to_mask import aggregate_cam_heatmaps, cam_to_pseudo_mask, normalize_min_max
from pseudo.visualization import save_mask, save_overlay, tensor_to_pil

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Generate pseudo masks from Grad-CAM with DenseCRF")
    parser.add_argument("--data-root", type=Path, default=ROOT.parent / "FracAtlas")
    parser.add_argument("--csv-path", type=Path, default=None)
    parser.add_argument("--image-root", type=Path, default=None)
    parser.add_argument("--checkpoint", type=Path, default=ROOT / "outputs" / "classifier" / "best.pt")
    parser.add_argument("--target-columns", type=str, default="hand,leg,hip,shoulder")
    parser.add_argument("--image-size", type=int, default=512)
    parser.add_argument("--batch-size", type=int, default=1)
    parser.add_argument("--num-workers", type=int, default=2)
    parser.add_argument("--output-dir", type=Path, default=ROOT / "outputs" / "pseudo_masks")
    parser.add_argument("--percentile", type=float, default=80.0)
    parser.add_argument("--min-area", type=int, default=200)
    parser.add_argument("--kernel-size", type=int, default=5)
    parser.add_argument("--fusion", type=str, default="weighted_mean", choices=("weighted_mean", "max"))
    parser.add_argument("--use-clahe", action="store_true")
    parser.add_argument("--crf-iter", type=int, default=5)
    return parser.parse_args()

def load_classifier(checkpoint_path: Path, num_classes: int, device: torch.device) -> DenseNet121AnatomyClassifier:
    model = DenseNet121AnatomyClassifier(num_classes=num_classes, pretrained=False)
    state_dict = torch.load(checkpoint_path, map_location="cpu")
    model.load_state_dict(state_dict["model_state_dict"], strict=True)
    model.to(device)
    model.eval()
    return model

def apply_dense_crf(image_np: np.ndarray, prob_mask: np.ndarray, iter_num: int = 5) -> np.ndarray:
    H, W = prob_mask.shape
    U = np.stack([1.0 - prob_mask, prob_mask], axis=0)
    U = U + 1e-5
    U = U / U.sum(axis=0, keepdims=True)
    
    unary = unary_from_softmax(U)
    unary = np.ascontiguousarray(unary)
    image_np = np.ascontiguousarray(image_np)
    
    d = dcrf.DenseCRF2D(W, H, 2)
    d.setUnaryEnergy(unary)
    
    d.addPairwiseEnergy(create_pairwise_gaussian(sdims=(3, 3), shape=(H, W)), compat=3)
    d.addPairwiseEnergy(
        create_pairwise_bilateral(sdims=(50, 50), srgb=(13, 13, 13), rgbim=image_np, compat=10),
        compat=10
    )
    
    Q = d.inference(iter_num)
    map_soln = np.argmax(Q, axis=0).reshape((H, W))
    return (map_soln * 255).astype(np.uint8)

def main() -> None:
    args = parse_args()
    csv_path = args.csv_path or (args.data_root / "dataset.csv")
    image_root = args.image_root or (args.data_root / "images")
    target_columns = [column.strip() for column in args.target_columns.split(",") if column.strip()]

    dataset = FracAtlasClassificationDataset(
        csv_path=csv_path,
        image_roots=image_root,
        target_columns=target_columns,
        image_size=args.image_size,
        augment=False,
        use_clahe=args.use_clahe,
    )
    loader = DataLoader(dataset, batch_size=args.batch_size, shuffle=False, num_workers=args.num_workers)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = load_classifier(args.checkpoint, num_classes=len(target_columns), device=device)
    gradcam = GradCAM(model, model.features.denseblock4, device=device)

    mask_dir = args.output_dir / "masks"
    overlay_dir = args.output_dir / "overlays"
    mask_dir.mkdir(parents=True, exist_ok=True)
    overlay_dir.mkdir(parents=True, exist_ok=True)

    for images, _, image_names in tqdm(loader, desc="pseudo-masks"):
        images = images.to(device)

        for index, image_name in enumerate(image_names):
            image_tensor = images[index : index + 1]
            with torch.no_grad():
                logits = model(image_tensor)
                class_weights = torch.sigmoid(logits)[0].detach().cpu().numpy()

            per_class_cams: list[np.ndarray] = []
            class_names = target_columns
            for class_index, class_name in enumerate(class_names):
                cam_output = gradcam.cam_for_class(image_tensor, class_index=class_index)
                per_class_cams.append(cam_output.cam[0].detach().cpu().numpy())
                save_overlay(
                    tensor_to_pil(image_tensor[0].detach().cpu()),
                    normalize_min_max(per_class_cams[-1]),
                    overlay_dir / f"{Path(image_name).stem}_{class_name}.png",
                )

            aggregated_cam = aggregate_cam_heatmaps(per_class_cams, weights=class_weights, mode=args.fusion)
            
            image_pil = tensor_to_pil(image_tensor[0].detach().cpu())
            image_np_uint8 = np.array(image_pil)
            prob_mask = normalize_min_max(aggregated_cam)
            
            refined_mask = apply_dense_crf(image_np_uint8, prob_mask, iter_num=args.crf_iter)

            mask_path = mask_dir / f"{Path(image_name).stem}.png"
            overlay_path = overlay_dir / f"{Path(image_name).stem}_aggregated.png"
            
            save_mask(refined_mask, mask_path) 
            save_overlay(image_pil, aggregated_cam, overlay_path)

    gradcam.close()

if __name__ == "__main__":
    main()

In [ ]:
!python generate_pseudo_masks.py \
  --data-root /root/.cache/kagglehub/datasets/mahmudulhasantasin/fracatlas-original-dataset/versions/1 \
  --checkpoint /content/drive/MyDrive/ThesisOutputs/classifier/best.pt \
  --target-columns hand,leg,hip,shoulder \
  --image-size 384 \
  --batch-size 1 \
  --num-workers 2 \
  --crf-iter 5 \
  --use-clahe \
  --output-dir /content/drive/MyDrive/ThesisOutputs/pseudo_masks

In [ ]:
%%writefile train_segmentation.py
from __future__ import annotations

import argparse
import csv
import sys
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from tqdm import tqdm

ROOT = Path(__file__).resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from config import SegmentationConfig
from datasets.fracatlas import FracAtlasSegmentationDataset, build_train_val_indices
from models.losses import dice_coefficient, iou_score
from models.unet import UNet

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Train U-Net on refined pseudo masks")
    parser.add_argument("--data-root", type=Path, default=ROOT.parent / "FracAtlas")
    parser.add_argument("--image-root", type=Path, default=None)
    parser.add_argument("--mask-root", type=Path, default=ROOT / "outputs" / "pseudo_masks" / "masks")
    parser.add_argument("--image-size", type=int, default=SegmentationConfig.image_size)
    parser.add_argument("--batch-size", type=int, default=SegmentationConfig.batch_size)
    parser.add_argument("--lr", type=float, default=SegmentationConfig.lr)
    parser.add_argument("--weight-decay", type=float, default=SegmentationConfig.weight_decay)
    parser.add_argument("--epochs", type=int, default=SegmentationConfig.epochs)
    parser.add_argument("--val-fraction", type=float, default=SegmentationConfig.val_fraction)
    parser.add_argument("--seed", type=int, default=SegmentationConfig.seed)
    parser.add_argument("--num-workers", type=int, default=4)
    parser.add_argument("--output-dir", type=Path, default=ROOT / "outputs" / "segmentation")
    parser.add_argument("--use-clahe", action="store_true")
    parser.add_argument("--focal-gamma", type=float, default=2.0)
    return parser.parse_args()

def seed_everything(seed: int) -> None:
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    import random
    import numpy as np
    random.seed(seed)
    np.random.seed(seed)

def focal_dice_loss(logits: torch.Tensor, targets: torch.Tensor, alpha: float = 0.25, gamma: float = 2.0) -> torch.Tensor:
    bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
    pt = torch.exp(-bce_loss)
    focal_loss = alpha * (1 - pt) ** gamma * bce_loss
    focal_loss = focal_loss.mean()
    
    probs = torch.sigmoid(logits)
    smooth = 1e-6
    intersection = (probs * targets).sum(dim=(2, 3))
    union = probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
    dice_score = (2.0 * intersection + smooth) / (union + smooth)
    dice_loss = 1.0 - dice_score.mean()
    
    return focal_loss + dice_loss

def run_epoch(model, loader, optimizer, device, train: bool, gamma: float) -> tuple[float, dict[str, float]]:
    total_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0
    batches = 0
    if train:
        model.train()
    else:
        model.eval()

    scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")
    progress = tqdm(loader, desc="train" if train else "val", leave=False)
    for images, masks, _ in progress:
        images = images.to(device)
        masks = masks.to(device)

        with torch.set_grad_enabled(train):
            with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
                logits = model(images)
                loss = focal_dice_loss(logits, masks, gamma=gamma)

            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        dice = dice_coefficient(logits.detach(), masks.detach())
        iou = iou_score(logits.detach(), masks.detach())
        total_loss += loss.item()
        total_dice += dice.item()
        total_iou += iou.item()
        batches += 1
        progress.set_postfix(loss=loss.item(), dice=dice.item(), iou=iou.item())

    if batches == 0:
        return 0.0, {"dice": 0.0, "iou": 0.0}
    return total_loss / batches, {"dice": total_dice / batches, "iou": total_iou / batches}

def save_checkpoint(path: Path, model: nn.Module, optimizer: torch.optim.Optimizer, epoch: int, best_metric: float) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_metric": best_metric,
    }, path)

def main() -> None:
    args = parse_args()
    seed_everything(args.seed)

    image_root = args.image_root or (args.data_root / "images")
    dataset = FracAtlasSegmentationDataset(
        image_roots=image_root,
        mask_root=args.mask_root,
        image_size=args.image_size,
        augment=True,
        use_clahe=args.use_clahe,
    )
    train_indices, val_indices = build_train_val_indices(len(dataset), val_fraction=args.val_fraction, seed=args.seed)
    train_dataset = Subset(dataset, train_indices)
    val_dataset = Subset(dataset, val_indices)

    train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True, num_workers=args.num_workers, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=args.batch_size, shuffle=False, num_workers=args.num_workers, pin_memory=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = UNet(in_channels=3, out_channels=1, base_channels=32).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)

    history_path = args.output_dir / "training_log.csv"
    best_val_loss = float("inf")
    args.output_dir.mkdir(parents=True, exist_ok=True)
    with history_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(["epoch", "train_loss", "train_dice", "train_iou", "val_loss", "val_dice", "val_iou"])

    for epoch in range(1, args.epochs + 1):
        train_loss, train_metrics = run_epoch(model, train_loader, optimizer, device, train=True, gamma=args.focal_gamma)
        val_loss, val_metrics = run_epoch(model, val_loader, optimizer, device, train=False, gamma=args.focal_gamma)

        with history_path.open("a", newline="", encoding="utf-8") as handle:
            writer = csv.writer(handle)
            writer.writerow([epoch, train_loss, train_metrics["dice"], train_metrics["iou"], val_loss, val_metrics["dice"], val_metrics["iou"]])

        print(f"Epoch {epoch:03d} | train_loss={train_loss:.4f} train_dice={train_metrics['dice']:.4f} val_loss={val_loss:.4f} val_dice={val_metrics['dice']:.4f}")

        save_checkpoint(args.output_dir / "last.pt", model, optimizer, epoch, best_val_loss)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_checkpoint(args.output_dir / "best.pt", model, optimizer, epoch, best_val_loss)

if __name__ == "__main__":
    main()

In [ ]:
!python train_segmentation.py \
  --data-root /root/.cache/kagglehub/datasets/mahmudulhasantasin/fracatlas-original-dataset/versions/1 \
  --mask-root /content/drive/MyDrive/ThesisOutputs/pseudo_masks/masks \
  --image-size 384 \
  --batch-size 8 \
  --num-workers 4 \
  --epochs 15 \
  --focal-gamma 2.0 \
  --use-clahe \
  --output-dir /content/drive/MyDrive/ThesisOutputs/segmentation

In [ ]:
%%writefile evaluate.py
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, SequentialSampler
from tqdm import tqdm

ROOT = Path(__file__).resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from datasets.fracatlas import FracAtlasSegmentationDataset
from models.losses import dice_coefficient, iou_score
from models.unet import UNet

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Evaluate U-Net Segmentation Model on Ground Truth")
    parser.add_argument("--data-root", type=Path, default=ROOT.parent / "FracAtlas")
    parser.add_argument("--image-root", type=Path, default=None)
    parser.add_argument("--gt-mask-root", type=Path, required=True)
    parser.add_argument("--segmentation-checkpoint", type=Path, required=True)
    parser.add_argument("--image-size", type=int, default=384)
    parser.add_argument("--batch-size", type=int, default=4)
    parser.add_argument("--num-workers", type=int, default=2)
    parser.add_argument("--output-dir", type=Path, default=ROOT / "outputs" / "evaluation")
    parser.add_argument("--use-clahe", action="store_true")
    return parser.parse_args()

def main() -> None:
    args = parse_args()
    args.output_dir.mkdir(parents=True, exist_ok=True)
    image_root = args.image_root or (args.data_root / "images")
    
    dataset = FracAtlasSegmentationDataset(
        image_roots=image_root,
        mask_root=args.gt_mask_root,
        image_size=args.image_size,
        augment=False,
        use_clahe=args.use_clahe,
    )
    loader = DataLoader(dataset, batch_size=args.batch_size, sampler=SequentialSampler(dataset), num_workers=args.num_workers)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = UNet(in_channels=3, out_channels=1, base_channels=32)
    checkpoint = torch.load(args.segmentation_checkpoint, map_location="cpu")
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    model.to(device)
    model.eval()

    total_dice, total_iou, batches = 0.0, 0.0, 0


    with torch.no_grad():
        for images, masks, _ in tqdm(loader, desc="Evaluating"):
            images = images.to(device)
            masks = masks.to(device)
            logits = model(images)
            
            total_dice += dice_coefficient(logits, masks).item()
            total_iou += iou_score(logits, masks).item()
            batches += 1

    final_dice = total_dice / batches
    final_iou = total_iou / batches

    print("\n" + "="*40)
    print("KẾT QUẢ ĐÁNH GIÁ TRÊN NHÃN CHUẨN ĐỘC LẬP")
    print("="*40)
    print(f"Mean Dice Score : {final_dice:.4f}")
    print(f"Mean IoU Score  : {final_iou:.4f}")
    print("="*40)

    with open(args.output_dir / "metrics_report.txt", "w") as f:
        f.write(f"Mean Dice Score: {final_dice:.4f}\n")
        f.write(f"Mean IoU Score: {final_iou:.4f}\n")

if __name__ == "__main__":
    main()

In [ ]:
!python evaluate.py \
  --data-root /root/.cache/kagglehub/datasets/mahmudulhasantasin/fracatlas-original-dataset/versions/1 \
  --gt-mask-root /root/.cache/kagglehub/datasets/mahmudulhasantasin/fracatlas-original-dataset/versions/1/Masks \
  --segmentation-checkpoint /content/drive/MyDrive/ThesisOutputs/segmentation/best.pt \
  --image-size 384 \
  --batch-size 4 \
  --use-clahe \
  --output-dir /content/drive/MyDrive/ThesisOutputs/evaluation